In [1]:
import os
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = "cross_validation/"

In [3]:
def get_dataset_data(directory):
    sizes = list()
    for file in os.listdir(directory):
        path = os.path.join(directory, file)
        if os.path.isfile(path):
            df = pd.read_csv(path, sep=" ", header=None)
            out = {"name": Path(path).stem,
                   "train_size": len(df) * 8 // 10
                   }
            sizes.append(out)
    df = pd.DataFrame(sizes)
    return df

In [4]:
size_df = get_dataset_data("../../contree/datasets")
size_df

,name,train_size
0,bank,1097
1,magic,15216
2,segment,1848
3,bean,10888
4,fault,1552
5,wilt,3871
6,raisin,720
7,avila,16693
8,occupancy,16448
9,skin,196045


In [5]:
all_df = list()
for file in os.listdir(path=BASE_DIR):
    path = os.path.join(BASE_DIR, file)
    name = Path(path).stem
    print(name)
    if os.path.isfile(path) and "cv" in name:
        df = pd.read_csv(path)
        # df["train_size"] =
        all_df.append(df)
df = pd.concat(all_df, ignore_index=True)
print(df.columns)
df = df.merge(size_df, on="name", how="left")
df["mean_test_acc"] = round(df["mean_test_acc"], 3)
df.to_csv(f"{BASE_DIR}/compiled.csv", index=None)

cv_contree
cv_c45
cv_lds
contree
lds
compiled
Index(['name', 'depth', 'mean_train_acc', 'mean_test_acc', 'mean_test_std',
       'method'],
      dtype='object')


In [8]:
depth = 6
df_pivot = df[df.depth == depth].pivot(index=['name', 'train_size'], columns='method', values='mean_test_acc')
df_pivot.reset_index(inplace=True)
def bold_max(row):
    # Identify the columns that are methods (exclude 'name' and 'train_size')
    method_cols = [c for c in df_pivot.columns if c not in ['name', 'train_size']]
    max_val = row[method_cols].max()

    for col in method_cols:
        # Format to 3 decimal places; bold if it matches max
        if row[col] == max_val:
            row[col] = f"\\textbf{{{row[col]:.3f}}}"
        else:
            row[col] = f"{row[col]:.3f}"
    return row

# Apply formatting
df_final = df_pivot.apply(bold_max, axis=1)
df_final.sort_values(["train_size"], ascending=False, inplace=True)
latex_output = df_final.to_latex(index=False, escape=False, column_format='llcc')
print(latex_output)

\begin{tabular}{llcc}
\toprule
name & train_size & C4.5 & Contree & Contree-LDS \\
\midrule
skin & 196045 & 0.989 & 0.991 & \textbf{0.997} \\
avila & 16693 & 0.656 & 0.627 & \textbf{0.709} \\
occupancy & 16448 & 0.989 & \textbf{0.992} & 0.991 \\
magic & 15216 & 0.841 & 0.716 & \textbf{0.854} \\
htru & 14318 & \textbf{0.977} & 0.906 & 0.976 \\
eeg & 11984 & 0.722 & 0.711 & \textbf{0.785} \\
bean & 10888 & 0.897 & 0.582 & \textbf{0.908} \\
room & 8103 & \textbf{0.996} & \textbf{0.996} & 0.995 \\
bidding & 5056 & 0.995 & 0.995 & \textbf{0.997} \\
page & 4378 & \textbf{0.967} & 0.957 & \textbf{0.967} \\
wilt & 3871 & \textbf{0.980} & 0.978 & 0.977 \\
rice & 3048 & \textbf{0.915} & 0.890 & 0.900 \\
segment & 1848 & 0.937 & 0.739 & \textbf{0.947} \\
fault & 1552 & 0.673 & 0.626 & \textbf{0.734} \\
bank & 1097 & 0.983 & \textbf{0.988} & 0.980 \\
raisin & 720 & \textbf{0.841} & 0.806 & 0.838 \\
\bottomrule
\end{tabular}

